## Substitution Matrix

This is the helper for generating a simple substitution matrix: all matches and mismatches are given the same score; it's not specific to different symbols (though such matrices can be created manually). 

In [1]:
def make_substitution_matrix(alphabet, match_score, mismatch_score):
    return {
        letter1: {
            letter2: match_score if letter1 == letter2 else mismatch_score
            for letter2 in alphabet
        }
        for letter1 in alphabet
    }

In [2]:
substitution_matrix = make_substitution_matrix("ACGT", 1, -2)
substitution_matrix

{'A': {'A': 1, 'C': -2, 'G': -2, 'T': -2},
 'C': {'A': -2, 'C': 1, 'G': -2, 'T': -2},
 'G': {'A': -2, 'C': -2, 'G': 1, 'T': -2},
 'T': {'A': -2, 'C': -2, 'G': -2, 'T': 1}}

## Move Choice

In [3]:
def choose_best_move(diagonal, up, left):
    if diagonal >= up and diagonal >= left:
        return diagonal, "diagonal"
    return (up, "up") if up >= left else (left, "left")

In [4]:
choose_best_move(-2, -4, -4)

(-2, 'diagonal')

## Matrix Filling

In [5]:
def fill_matrix(seq1, seq2, substitution_matrix, gap_cost, mode):
    rows, columns = len(seq1) + 1, len(seq2) + 1
    score_matrix = [[0] * columns for _ in range(rows)]
    trace_matrix = [[""] * columns for _ in range(rows)]

    if mode == "global":
        for i in range(1, rows):
            trace_matrix[i][0] = "up"
            score_matrix[i][0] = -i * gap_cost
        for j in range(1, columns):
            trace_matrix[0][j] = "left"
            score_matrix[0][j] = -j * gap_cost

    best_i = best_j = best_score = 0

    for i in range(1, rows):
        for j in range(1, columns):
            diagonal = (
                score_matrix[i - 1][j - 1]
                + substitution_matrix[seq1[i - 1]][seq2[j - 1]]
            )
            up = score_matrix[i - 1][j] - gap_cost
            left = score_matrix[i][j - 1] - gap_cost
            score, direction = choose_best_move(diagonal, up, left)

            if mode == "local" and score <= 0:
                score, direction = 0, ""

            score_matrix[i][j], trace_matrix[i][j] = score, direction

            if mode == "local" and score > best_score:
                best_i, best_j, best_score = i, j, score

    if mode == "global":
        return score_matrix, trace_matrix, rows - 1, columns - 1, score_matrix[-1][-1]
    return score_matrix, trace_matrix, best_i, best_j, best_score

In [6]:
global_score_matrix, global_trace_matrix, global_i, global_j, global_score = fill_matrix(
    "AGGACT",
    "GTGAGT",
    substitution_matrix,
    2,
    "global"
)
global_score_matrix

[[0, -2, -4, -6, -8, -10, -12],
 [-2, -2, -4, -6, -5, -7, -9],
 [-4, -1, -3, -3, -5, -4, -6],
 [-6, -3, -3, -2, -4, -4, -6],
 [-8, -5, -5, -4, -1, -3, -5],
 [-10, -7, -7, -6, -3, -3, -5],
 [-12, -9, -6, -8, -5, -5, -2]]

In [7]:
local_score_matrix, local_trace_matrix, local_i, local_j, local_score = fill_matrix(
    "AGGACT",
    "GTGAGT",
    substitution_matrix,
    2,
    "local"
)
local_score_matrix

[[0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0],
 [0, 1, 0, 1, 0, 2, 0],
 [0, 1, 0, 1, 0, 1, 0],
 [0, 0, 0, 0, 2, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0, 0, 1]]

## Traceback

In [8]:
def traceback(seq1, seq2, trace_matrix, i, j, mode):
    aligned1, aligned2 = [], []

    while i > 0 or j > 0:
        if mode == "local" and (i == 0 or j == 0):
            break

        direction = trace_matrix[i][j]
        if direction == "diagonal":
            aligned1.append(seq1[i - 1])
            aligned2.append(seq2[j - 1])
            i -= 1
            j -= 1
        elif direction == "up":
            aligned1.append(seq1[i - 1])
            aligned2.append("-")
            i -= 1
        elif direction == "left":
            aligned1.append("-")
            aligned2.append(seq2[j - 1])
            j -= 1
        else:
            break

    aligned1.reverse()
    aligned2.reverse()
    return "".join(aligned1), "".join(aligned2)

In [9]:
traceback("AGGACT", "GTGAGT", global_trace_matrix, global_i, global_j, "global")

('AG-GACT', '-GTGAGT')

In [10]:
traceback("AGGACT", "GTGAGT", local_trace_matrix, local_i, local_j, "local")

('AG', 'AG')

## Alignment Functions

In [11]:
def align(seq1, seq2, substitution_matrix, gap_cost, mode):
    _, trace_matrix, i, j, score = fill_matrix(seq1, seq2, substitution_matrix, gap_cost, mode)
    return (*traceback(seq1, seq2, trace_matrix, i, j, mode), score)


def needleman_wunsch(seq1, seq2, substitution_matrix, gap_cost):
    return align(seq1, seq2, substitution_matrix, gap_cost, "global")


def smith_waterman(seq1, seq2, substitution_matrix, gap_cost):
    return align(seq1, seq2, substitution_matrix, gap_cost, "local")

In [12]:
needleman_wunsch("AGGACT", "GTGAGT", substitution_matrix, 2)

('AG-GACT', '-GTGAGT', -2)

In [13]:
smith_waterman("AGGACT", "GTGAGT", substitution_matrix, 2)

('AG', 'AG', 2)

## Testing Helper

In [14]:
def get_match_line(aligned1, aligned2):
    match_line = []
    for letter1, letter2 in zip(aligned1, aligned2):
        if letter1 == "-" or letter2 == "-":
            match_line.append(" ")
        elif letter1 == letter2:
            match_line.append("|")
        else:
            match_line.append(".")
    return "".join(match_line)


def display_alignment(aligned1, aligned2, score):
    print(aligned1)
    print(get_match_line(aligned1, aligned2))
    print(aligned2)
    print(f"Score: {score}")


def run_example(name, seq1, seq2, alphabet, match_score, mismatch_score, gap_cost):
    substitution_matrix = make_substitution_matrix(alphabet, match_score, mismatch_score)

    print("=" * 72)
    print(name)
    print(f"Sequence 1: {seq1}")
    print(f"Sequence 2: {seq2}")
    print(f"Gap cost: {gap_cost}")
    print()

    print("Needleman-Wunsch global alignment")
    display_alignment(*needleman_wunsch(seq1, seq2, substitution_matrix, gap_cost))
    print()

    print("Smith-Waterman local alignment")
    display_alignment(*smith_waterman(seq1, seq2, substitution_matrix, gap_cost))

## Test Inputs

In [15]:
examples = [
    (
        "Example 1: short DNA sequences from the assignment",
        "AGGACT",
        "GTGAGT",
        "ACGT",
        1,
        -2,
        2
    ),
    (
        "Example 2: one internal deletion",
        "ACGTCGTA",
        "ACGCGTA",
        "ACGT",
        2,
        -1,
        2
    ),
    (
        "Example 3: shared high-scoring local motif",
        "TTTACCGTACGG",
        "GGGACCGTAAAC",
        "ACGT",
        3,
        -3,
        2
    ),
    (
        "Example 4: different simple DNA scoring matrix",
        "GATTACA",
        "GCATGCA",
        "ACGT",
        2,
        -1,
        2
    ),
    (
        "Example 5: protein-like sequences",
        "HEAGAWGHEE",
        "PAWHEAE",
        "AEGHPW",
        2,
        -1,
        2
    )
]

## Example 1

In [16]:
run_example(*examples[0])

Example 1: short DNA sequences from the assignment
Sequence 1: AGGACT
Sequence 2: GTGAGT
Gap cost: 2

Needleman-Wunsch global alignment
AG-GACT
 | ||.|
-GTGAGT
Score: -2

Smith-Waterman local alignment
AG
||
AG
Score: 2


## Example 2

In [17]:
run_example(*examples[1])

Example 2: one internal deletion
Sequence 1: ACGTCGTA
Sequence 2: ACGCGTA
Gap cost: 2

Needleman-Wunsch global alignment
ACGTCGTA
||| ||||
ACG-CGTA
Score: 12

Smith-Waterman local alignment
ACGTCGTA
||| ||||
ACG-CGTA
Score: 12


## Example 3

In [18]:
run_example(*examples[2])

Example 3: shared high-scoring local motif
Sequence 1: TTTACCGTACGG
Sequence 2: GGGACCGTAAAC
Gap cost: 2

Needleman-Wunsch global alignment
TTTACCGT--ACGG
...|||||  ||  
GGGACCGTAAAC--
Score: 4

Smith-Waterman local alignment
ACCGTA
||||||
ACCGTA
Score: 18


## Example 4

In [19]:
run_example(*examples[3])

Example 4: different simple DNA scoring matrix
Sequence 1: GATTACA
Sequence 2: GCATGCA
Gap cost: 2

Needleman-Wunsch global alignment
GATTACA
|..|.||
GCATGCA
Score: 5

Smith-Waterman local alignment
TACA
|.||
TGCA
Score: 5


## Example 5

In [20]:
run_example(*examples[4])

Example 5: protein-like sequences
Sequence 1: HEAGAWGHEE
Sequence 2: PAWHEAE
Gap cost: 2

Needleman-Wunsch global alignment
HEAGAWGHE-E
   .|| || |
---PAW-HEAE
Score: -1

Smith-Waterman local alignment
HEA
|||
HEA
Score: 6


## Manual Workthrough

### Needleman-Wunsch example

The first example aligns:

$$
x=\texttt{AGGACT}
$$

$$
y=\texttt{GTGAGT}
$$

The scoring scheme is:

$$
s(a,b)=
\begin{cases}
1, & \text{if } a=b \\
-2, & \text{if } a\ne b
\end{cases}
\quad\text{and}\quad g=2
$$

For global alignment, leading and trailing gaps are part of the score. Therefore, the first row and first column contain accumulated gap penalties.

For cell $(1,1)$, the letters are `A` and `G`:

$$
d_{1,1}=0-2=-2,\quad u_{1,1}=-2-2=-4,\quad l_{1,1}=-2-2=-4
$$

$$
H_{1,1}=\max(-2,-4,-4)=-2
$$

For cell $(2,1)$, the letters are `G` and `G`:

$$
d_{2,1}=-2+1=-1,\quad u_{2,1}=-2-2=-4,\quad l_{2,1}=-4-2=-6
$$

$$
H_{2,1}=\max(-1,-4,-6)=-1
$$

After applying the same rule to every cell, the Needleman-Wunsch score matrix is:

|  | - | G | T | G | A | G | T |
|--|--:|--:|--:|--:|--:|--:|--:|
| - | 0 | -2 | -4 | -6 | -8 | -10 | -12 |
| A | -2 | -2 | -4 | -6 | -5 | -7 | -9 |
| G | -4 | -1 | -3 | -3 | -5 | -4 | -6 |
| G | -6 | -3 | -3 | -2 | -4 | -4 | -6 |
| A | -8 | -5 | -5 | -4 | -1 | -3 | -5 |
| C | -10 | -7 | -7 | -6 | -3 | -3 | -5 |
| T | -12 | -9 | -6 | -8 | -5 | -5 | -2 |

The traceback starts in the bottom-right cell because the full sequences must be aligned:

| Step | Cell | Direction | Alignment column |
|--:|--|--|--|
| 1 | $(6,6)$ | D | `T`/`T` |
| 2 | $(5,5)$ | D | `C`/`G` |
| 3 | $(4,4)$ | D | `A`/`A` |
| 4 | $(3,3)$ | D | `G`/`G` |
| 5 | $(2,2)$ | L | `-`/`T` |
| 6 | $(2,1)$ | D | `G`/`G` |
| 7 | $(1,0)$ | U | `A`/`-` |

The final global alignment is:

```text
AG-GACT
 | ||.|
-GTGAGT
Score: -2
```

The score can be checked directly from the alignment:

$$
-2 + 1 - 2 + 1 + 1 - 2 + 1 = -2
$$

### Smith-Waterman example

Smith-Waterman uses the same sequences and scoring scheme, but every cell is also compared to zero. Negative paths are discarded because a local alignment may start later in the sequences.

For cell $(1,1)$, the letters are `A` and `G`:

$$
d_{1,1}=0-2=-2,\quad u_{1,1}=0-2=-2,\quad l_{1,1}=0-2=-2
$$

$$
H_{1,1}=\max(0,-2,-2,-2)=0
$$

For cell $(1,4)$, the letters are `A` and `A`:

$$
d_{1,4}=0+1=1,\quad u_{1,4}=0-2=-2,\quad l_{1,4}=0-2=-2
$$

$$
H_{1,4}=\max(0,1,-2,-2)=1
$$

The complete Smith-Waterman score matrix is:

|  | - | G | T | G | A | G | T |
|--|--:|--:|--:|--:|--:|--:|--:|
| - | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| A | 0 | 0 | 0 | 0 | 1 | 0 | 0 |
| G | 0 | 1 | 0 | 1 | 0 | 2 | 0 |
| G | 0 | 1 | 0 | 1 | 0 | 1 | 0 |
| A | 0 | 0 | 0 | 0 | 2 | 0 | 0 |
| C | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| T | 0 | 0 | 1 | 0 | 0 | 0 | 1 |

The traceback starts at the first best-scoring cell found by the program, which is $(2,5)$ with score 2:

| Step | Cell | Direction | Alignment column |
|--:|--|--|--|
| 1 | $(2,5)$ | D | `G`/`G` |
| 2 | $(1,4)$ | D | `A`/`A` |

The final local alignment is:

```text
AG
||
AG
Score: 2
```